# Preprocessing Data Figures 4 and 5: 4D-Var State Estimation and Analysis Skill

In [1]:
%matplotlib inline
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns

import numpy as np
import pandas as pd
import glob

import joblib


In [ ]:
# Define parameters
min_val = "0.05"
suffix = "_frozen_params_5_384_compressed.pkl"  # or "" if not frozen

# Path to data (user downloads from Zenodo)
base_dir = "../data/assimilation_results/"

# Templates for each dataset
templates = {
    "baseline": f"two_samples_per_day_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
    "irreg_sparse": f"irregular_sparse_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
    "field_informed": f"field_informed_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
}

# Load dynamically
xb_dicts = {}
for name, fname in templates.items():
    fpath = os.path.join(base_dir, fname)
    if os.path.exists(fpath):
        xb_dicts[name] = joblib.load(fpath)
    else:
        print(f"File not found: {fpath}")

rows = []

for dataset_name, xb_dict in xb_dicts.items():
    for i, record in xb_dict.items():
        perturbed = record["perturbed_initial_state"]
        truth = record["truth"][0]
        diff = perturbed - truth

        row = {
            "dataset": dataset_name,
            "index": i,
            "N_perturbed": perturbed[0],
            "P_perturbed": perturbed[1],
            "Z_perturbed": perturbed[2],
            "distance_N": np.abs(perturbed[0] - truth[0]),
            "distance_P": np.abs(perturbed[1] - truth[1]),
            "distance_Z": np.abs(perturbed[2] - truth[2]),
            "distance_total_L2": np.linalg.norm(perturbed - truth),
            "distance_total_L1": np.sum(np.abs(perturbed - truth)),
            "rk4_run_time": record["rk4_run_time"],
            "pinn_run_time": record["pi_npz_run_time"],
        }
        rows.append(row)

df_xb_distances = pd.DataFrame(rows)


In [3]:
# --- Collect predictions and truth ---
pinn_preds = []
rk4_truths = []

for i, record in xb_dict.items():
    pinn_preds.append(np.array(record["pi_npz_background_state"]))  # PINN forecast
    rk4_truths.append(np.array(record["rk4_background_state"]))     # RK4 truth

pinn_preds = np.array(pinn_preds)  # shape [N, T, 3]
rk4_truths = np.array(rk4_truths)  # shape [N, T, 3]

# --- Compute PINN skill vs RK4 truth ---
pinn_scores = 1 - (np.abs(pinn_preds - rk4_truths) / np.abs(rk4_truths))

# Mean/std over all trajectories (axis=0)
pinn_mean = np.nanmean(pinn_scores, axis=0).T  # shape (3, T)
pinn_std  = np.nanstd(pinn_scores, axis=0).T

# --- Summary statistics ---
skill_mean_over_time = np.nanmean(pinn_mean, axis=1)
skill_std_over_time  = np.nanmean(pinn_std, axis=1)

global_skill_mean = np.nanmean(pinn_mean)
global_skill_std  = np.nanmean(pinn_std)

traj_skill_means = np.nanmean(pinn_scores, axis=(1,2))
traj_skill_stds  = np.nanstd(pinn_scores, axis=(1,2))

early_skill = np.nanmean(pinn_mean[:, :50], axis=1)
late_skill  = np.nanmean(pinn_mean[:, -50:], axis=1)

summary_df = pd.DataFrame({
    "Variable": ["N", "P", "Z"],
    "Mean Skill": skill_mean_over_time,
    "Std Skill": skill_std_over_time,
    "Early Skill (first 50 steps)": early_skill,
    "Late Skill (last 50 steps)": late_skill
})


In [ ]:
# === Parameters you can toggle ===
min_val = "0.05"  # switch between "0.001", "0.05", etc.
suffix  = "_5_384_compressed.pkl"  # or "_compressed.pkl"
rk4_suffix  = "_compressed.pkl"  # or "_compressed.pkl"

# Path to data (user downloads from Zenodo)
base_path = "../data/assimilation_results/"

# === Template builder ===
file_info = [
    ("baseline",      "PI-NPZ",          f"two_samples_per_day_pi_npz_frozen_params_assimilation_10000xb_estimates_min_guess_{min_val}_2000{suffix}"),
    ("baseline",      "Traditional-NPZ", f"two_samples_per_day_rk4_assimilation_10000xb_estimates_min_guess_{min_val}_2000{rk4_suffix}"),

    ("irreg_sparse",  "PI-NPZ",          f"irregular_sparse_pi_npz_frozen_params_assimilation_10000xb_estimates_min_guess_{min_val}_2000{suffix}"),
    ("irreg_sparse",  "Traditional-NPZ", f"irregular_sparse_rk4_assimilation_10000xb_estimates_min_guess_{min_val}_2000{rk4_suffix}"),

    ("field_informed","PI-NPZ",          f"field_informed_pi_npz_frozen_assimilation_10000xb_estimates_min_guess_{min_val}_2000{suffix}"),
    ("field_informed","Traditional-NPZ", f"field_informed_rk4_assimilation_10000xb_estimates_min_guess_{min_val}_2000{rk4_suffix}"),
]

# === Map prefixes for consistent key lookup ===
prefix_map = {
    "PI-NPZ": "pi_npz",
    "Traditional-NPZ": "rk4",
}

# === Build DataFrame ===
records = []
for dataset_label, method, filename in file_info:
    path = os.path.join(base_path, filename)
    if not os.path.exists(path):
        print(f"Missing file: {path}")
        continue

    data = joblib.load(path)
    prefix = prefix_map[method]

    for i, result in data.items():
        record = {
            "dataset": dataset_label,
            "method": method,
            "index": i,
            "jo_b": result.get(f"jo_b_{prefix}"),
            "jb_b": result.get(f"jb_b_{prefix}"),
            "J_total_b": result.get(f"J_total_b_{prefix}"),
            "jo_a": result.get(f"jo_a_{prefix}"),
            "jb_a": result.get(f"jb_a_{prefix}"),
            "J_total_a": result.get(f"J_total_a_{prefix}"),
            "misfitb": result.get(f"{prefix}_misfitb"),
            "misfita": result.get(f"{prefix}_misfita"),
            "Improvement": result.get(f"Improvement_{prefix}", result.get("Improvement")),
            "pct_drop_J": result.get(f"pct_drop_J_{prefix}"),
            "pct_drop_Jo": result.get(f"pct_drop_Jo_{prefix}"),
            "pct_drop_Jb": result.get(f"pct_drop_Jb_{prefix}"),
            "cg_iterations": result.get(f"cg_iterations_{prefix}"),
            "converged": result.get(f"converged_{prefix}"),
            "exit_code": result.get(f"exit_code_{prefix}"),
            "runtime": result.get(f"{prefix}_time"),
            # NEW: store trajectories
            "xa": result.get(f"xa_{prefix}"),
            "xb": result.get(f"xb_{prefix}"),
        }
        records.append(record)

df_flat = pd.DataFrame.from_records(records)


In [8]:
# Add a log-transformed runtime column before melt
df_flat["log_runtime"] = np.log10(df_flat["runtime"])
df_flat["jo_jb_ratio"] = df_flat["jo_a"] / df_flat["jb_a"]
df_flat["log_jo_jb_ratio"] = np.log10(df_flat["jo_jb_ratio"])
df_flat["pct_drop_J_100"] = 100 * df_flat["pct_drop_J"]

# Update metrics list
metrics = ["log_runtime", "pct_drop_J_100", "log_jo_jb_ratio", "Improvement"]

# Melt for plotting
df_long = df_flat.melt(
    id_vars=["method", "dataset"],
    value_vars=metrics,
    var_name="metric",
    value_name="value"
)

# Nice labels
df_long["metric"] = df_long["metric"].replace({
    "log_runtime": "log10(Runtime)",
    "log_jo_jb_ratio": "log10(jo_a / jb_a)"
})

# === Keep only 3 metrics (drop Improvement) ===
metrics_normal = ["runtime", "pct_drop_J_100", "log_jo_jb_ratio"]

df_long_normal = df_flat.melt(
    id_vars=["method", "dataset"],
    value_vars=metrics_normal,
    var_name="metric",
    value_name="value"
)

# === Relabel metrics ===
df_long_normal["metric"] = df_long_normal["metric"].replace({
    "runtime": "Computational Time (s)",
    "pct_drop_J_100": "Cost Function Reduction (%)",
    "log_jo_jb_ratio": "log$_{10}(J_o/J_b)$"
})



In [13]:
# Save source data (recommended for data repository)
df_metrics = df_flat.drop(columns=['xa', 'xb'])
df_metrics.to_csv('./data/processed_assimilation_data/assimilation_metrics.csv', index=False)

# Optionally also save the plotting-ready format
df_long_normal.to_csv('./data/processed_assimilation_data/assimilation_metrics_long.csv', index=False)

In [ ]:
# Save initial condition perturbations and distances
df_xb_distances.to_csv('./data/processed_assimilation_data/background_state_distances.csv', index=False)

'/Users/egank31/Documents/Documents - Otto/pinn-4dvar-npz'

In [16]:
# === Get truth trajectory ===
example_dataset = list(xb_dicts.values())[0]
example_record = list(example_dataset.values())[0]
truth = example_record["truth"]

# === Compute aggregated skills (ONLY THIS) ===
aggregated_skills = {}
for method in ["PI-NPZ", "Traditional-NPZ"]:
    all_trajectories = []
    
    # Collect ALL xa trajectories for this method across all datasets
    for dataset_name in df_flat["dataset"].unique():
        subset = df_flat[(df_flat["dataset"] == dataset_name) & 
                        (df_flat["method"] == method)]
        xa_list = subset["xa"].dropna().values
        
        if len(xa_list) > 0:
            xa_array = np.stack(xa_list, axis=0)
            all_trajectories.append(xa_array)
    
    # Stack all trajectories across datasets
    all_xa = np.concatenate(all_trajectories, axis=0)
    
    # Compute skill vs truth
    truth_array = np.broadcast_to(truth, all_xa.shape)
    skill_scores = 1 - (np.abs(all_xa - truth_array) / np.abs(truth_array))
    
    # Aggregate over variables
    skill_per_traj = np.nanmean(skill_scores, axis=2)
    
    # Overall mean and std across all trajectories at each time
    skill_mean = np.nanmean(skill_per_traj, axis=0)
    skill_std = np.nanstd(skill_per_traj, axis=0)
    
    # Global skill for each trajectory (averaged over time)
    global_skill_per_traj = np.nanmean(skill_per_traj, axis=1)
    
    # Mean and std of global skill
    global_skill_mean = np.nanmean(global_skill_per_traj)
    global_skill_std = np.nanstd(global_skill_per_traj)
    
    aggregated_skills[method] = {
        "mean": skill_mean,
        "std": skill_std,
        "global_mean": global_skill_mean,
        "global_std": global_skill_std
    }

# === Save aggregated skills ===
skill_rows = []
for method in ["Traditional-NPZ", "PI-NPZ"]:
    skill_data = aggregated_skills[method]
    T = len(skill_data["mean"])
    times = np.linspace(0, 7, T)
    
    for t_idx, time_val in enumerate(times):
        skill_rows.append({
            "method": method,
            "time_days": time_val,
            "skill_mean": skill_data["mean"][t_idx],
            "skill_std": skill_data["std"][t_idx],
        })
    
    # Add global skill
    skill_rows.append({
        "method": method,
        "time_days": -1,  # marker for global aggregate
        "skill_mean": skill_data["global_mean"],
        "skill_std": skill_data["global_std"],
    })

df_skills_aggregated = pd.DataFrame(skill_rows)
df_skills_aggregated.to_csv('./data/processed_assimilation_data/analysis_skills_aggregated.csv', index=False)

# # Print results for text
# print("\nAnalysis forecast skill summary:")
# for method in ["Traditional-NPZ", "PI-NPZ"]:
#     mean = aggregated_skills[method]["global_mean"]
#     std = aggregated_skills[method]["global_std"]
#     print(f"{method}: {mean:.3f} ± {std:.3f} ({mean*100:.1f}% ± {std*100:.1f}%)")

# # Calculate difference
# trad_mean = aggregated_skills["Traditional-NPZ"]["global_mean"]
# pinn_mean = aggregated_skills["PI-NPZ"]["global_mean"]
# diff = pinn_mean - trad_mean
# print(f"\nDifference (PI-NPZ - Traditional): {diff:.4f} ({diff*100:.2f} percentage points)")
